# HR Analytics · Descriptive statistics
**1 Session · Data Science Academy · Statistics and Machine Learning class**

Run the cells one by one: click a cell, then ▶ (or Shift+Enter). The data loads automatically from GitHub.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/QssAnalytic/dsa-hr-analytics/main/data/HR_Analytics_data.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

## 1. Missing values for every column

In [ ]:
df.isna().sum()

## 2. Numerical variables
Constant columns (EmployeeCount, StandardHours), the ID and the pre-computed z-score are left out.

In [ ]:
num = df.select_dtypes('number').drop(columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'ZMonthlyIncome'])
print(len(num.columns), 'numerical variables')
list(num.columns)

## 3. Descriptive statistics
Min, max, mean, median, mode, P90, P99, std, Q1, Q3, skewness and (excess) kurtosis.

In [ ]:
summary = pd.DataFrame({
    'min': num.min(),
    'max': num.max(),
    'mean': num.mean(),
    'median': num.median(),
    'mode': num.mode().iloc[0],
    'Q1': num.quantile(0.25),
    'Q3': num.quantile(0.75),
    'P90': num.quantile(0.90),
    'P99': num.quantile(0.99),
    'std': num.std(),
    'skewness': num.skew(),
    'kurtosis': num.kurt(),
}).round(2)
summary

## 4. Histogram for each variable
Yellow = mean, blue dashed = median.

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(16, 20))
for ax, col in zip(axes.ravel(), num.columns):
    ax.hist(num[col], bins=30, color='#2E2E38', edgecolor='white')
    ax.axvline(num[col].mean(), color='#FFE600', lw=2.5)
    ax.axvline(num[col].median(), color='#188CE5', lw=1.5, ls='--')
    ax.set_title(f"{col}\nskew {num[col].skew():.2f} · kurt {num[col].kurt():.2f}", fontsize=10)
for ax in axes.ravel()[len(num.columns):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Boxplot for each feature and IQR outliers
Outlier = below Q1 − 1.5·IQR or above Q3 + 1.5·IQR.

In [ ]:
q1, q3 = num.quantile(0.25), num.quantile(0.75)
iqr = q3 - q1
iqr_outliers = ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum()

fig, axes = plt.subplots(6, 4, figsize=(16, 16))
for ax, col in zip(axes.ravel(), num.columns):
    ax.boxplot(num[col], vert=False, patch_artist=True,
               boxprops=dict(facecolor='#FFE600'), flierprops=dict(markerfacecolor='red', markersize=4))
    ax.set_title(f"{col} · {iqr_outliers[col]} IQR outliers", fontsize=10)
    ax.set_yticks([])
for ax in axes.ravel()[len(num.columns):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

iqr_outliers.sort_values(ascending=False)

## 6. Z-score of MonthlyIncome
z = (x − mean) / std. Outlier if |z| > 3.

In [ ]:
income = df['MonthlyIncome']
df['z_income'] = (income - income.mean()) / income.std()

print('|z| > 3:', (df['z_income'].abs() > 3).sum())
print('z > 2  :', (df['z_income'] > 2).sum())
print('max z  :', round(df['z_income'].max(), 2))
print('IQR outliers for comparison:', iqr_outliers['MonthlyIncome'])

df.sort_values('z_income', ascending=False)[['EmployeeNumber', 'JobRole', 'MonthlyIncome', 'z_income']].head(10)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df['z_income'], bins=40, color='#2E2E38', edgecolor='white')
for v, c in [(2, '#188CE5'), (-2, '#188CE5'), (3, 'red'), (-3, 'red')]:
    plt.axvline(v, color=c, ls='--')
plt.title('Z-scores of MonthlyIncome')
plt.show()

## 7. Normality: MonthlyIncome, Age, TotalWorkingYears
Shapiro–Wilk: H₀ = the data are normal. p < 0.05 → not normal.

In [ ]:
for col in ['MonthlyIncome', 'Age', 'TotalWorkingYears']:
    w, p = stats.shapiro(df[col])
    verdict = 'NOT normal' if p < 0.05 else 'normal'
    print(f"{col:18s} W = {w:.3f}  p = {p:.2e}  skew = {df[col].skew():.2f}  kurt = {df[col].kurt():.2f}  -> {verdict}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for j, col in enumerate(['MonthlyIncome', 'Age', 'TotalWorkingYears']):
    x = df[col]
    ax = axes[0, j]
    ax.hist(x, bins=30, density=True, color='#C4C4CD', edgecolor='white')
    grid = np.linspace(x.min(), x.max(), 300)
    ax.plot(grid, stats.norm.pdf(grid, x.mean(), x.std()), color='#FFE600', lw=3, label='normal curve')
    ax.plot(grid, stats.gaussian_kde(x)(grid), color='#2E2E38', lw=2, label='actual')
    ax.set_title(col); ax.legend()
    stats.probplot(x, dist='norm', plot=axes[1, j])
    axes[1, j].set_title(f'Q-Q plot · {col}')
plt.tight_layout()
plt.show()

## Next step
Save a copy to your Drive (**File → Save a copy in Drive**), then push it to your own GitHub portfolio.